In [ ]:
# Imports and Config

import yaml
from pathlib import Path
import pandas as pd
from src import preprocessing, modelling, utils

# Load configuration
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Paths
model_file = Path(config["paths"]["model_file"])

In [ ]:
# Define new run data

# Example new run data
new_run = pd.DataFrame({
    "run_id": [999],
    "distance_km": [15],
    "duration_min": [85],
    "avg_pace_min_km": [5.67],
    "effort_type": ["tempo"],
    "elevation_gain_m": [120],
    "temperature_c": [18]
})

# Map effort_score and distance category
new_run["effort_score"] = new_run["effort_type"].map({"easy":1,"tempo":2,"race":3})
new_run["distance_category"] = pd.cut(new_run["distance_km"], bins=[0,10,21.1,42.3], labels=["short","medium","long"])


In [ ]:
# Preprocessing

# Same preprocessing as training
new_run_prepared = preprocessing.preprocess_data(
    new_run,
    drop_na_columns=config["preprocessing"]["drop_na_columns"],
    normalize_columns=config["preprocessing"]["normalize_columns"]
)

utils.check_dataframe(new_run_prepared, "New Run Preprocessed")


In [ ]:
# Attempt Prediction

predicted_hr = modelling.predict_sustainable_hr(
    new_run_prepared,
    model_file=model_file,
    features=config["modelling"]["features"]
)

print(f"Predicted sustainable HR for this run: {predicted_hr.iloc[0]:.1f} bpm")


In [ ]:
# Display Predicted vs Features

import matplotlib.pyplot as plt
import seaborn as sns

sns.scatterplot(
    data=new_run_prepared,
    x="distance_km_norm",
    y=predicted_hr
)
plt.xlabel("Normalized Distance (km)")
plt.ylabel("Predicted Sustainable HR")
plt.title("Single Run Prediction")
plt.show()
